# Ch 10 — 폐암 수술 환자 생존율 예측

원본: `deep_code/01_My_First_Deeplearning.py`

다루는 내용:
1. CSV 로드 (헤더 없음)
2. 17 컬럼 → 1 컬럼 분리
3. Dense 30 → Dense 1 (sigmoid) 모델
4. 책의 MSE 손실 vs 표준 BCE 손실 비교
5. 출력값 직접 디코딩

## 0. 환경 / 시드

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
from keras import Input, Sequential
from keras.layers import Dense

keras.utils.set_random_seed(0)

print("Keras:", keras.__version__)

## 1. 데이터 로드

책은 `np.loadtxt` 를 사용했지만, pandas로 통일.

In [ ]:
DATA = "../../data/ThoraricSurgery.csv"
df = pd.read_csv(DATA, header=None)
print("shape:", df.shape)
df.head()

In [ ]:
X = df.iloc[:, 0:17].to_numpy(dtype="float32")
y = df.iloc[:, 17].to_numpy(dtype="float32")
print("X:", X.shape, "y:", y.shape)
print("y 분포:", np.bincount(y.astype(int)))

## 2. 모델 — Dense 30 → Dense 1 (sigmoid)

In [ ]:
def build_model():
    return Sequential([
        Input(shape=(17,)),
        Dense(30, activation="relu"),
        Dense(1, activation="sigmoid"),
    ])

model = build_model()
model.summary()

## 3. 학습 — 책 버전 (MSE 손실)

책의 컴파일 그대로. 이진 분류에 MSE를 쓰는 건 흔치 않지만 동작은 함.

In [ ]:
keras.utils.set_random_seed(0)
model_mse = build_model()
model_mse.compile(loss="mean_squared_error", optimizer="adam", metrics=["accuracy"])
hist_mse = model_mse.fit(X, y, epochs=30, batch_size=10, verbose=0)
print(f"MSE 모델 train accuracy: {hist_mse.history['accuracy'][-1]:.4f}")

## 4. 학습 — 표준 버전 (BCE 손실)

`binary_crossentropy` 가 이진 분류의 표준 손실. 같은 모델, 같은 데이터로 비교.

In [ ]:
keras.utils.set_random_seed(0)
model_bce = build_model()
model_bce.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
hist_bce = model_bce.fit(X, y, epochs=30, batch_size=10, verbose=0)
print(f"BCE 모델 train accuracy: {hist_bce.history['accuracy'][-1]:.4f}")

## 5. 학습 곡선 비교

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(hist_mse.history["loss"], label="MSE loss")
ax[0].plot(hist_bce.history["loss"], label="BCE loss")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[0].set_title("Loss curves")

ax[1].plot(hist_mse.history["accuracy"], label="MSE")
ax[1].plot(hist_bce.history["accuracy"], label="BCE")
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("accuracy"); ax[1].legend(); ax[1].grid(alpha=0.3)
ax[1].set_title("Train accuracy")
plt.tight_layout(); plt.show()

## 6. 출력값 직접 디코딩

`model.predict()` 는 `(N, 1)` 시그모이드 확률을 반환. 0.5 임계값으로 클래스 변환하고 직접 정확도 확인.

In [ ]:
probs = model_bce.predict(X[:10], verbose=0)
preds = (probs > 0.5).astype("int32").flatten()
print("확률(처음 10개):", probs.flatten().round(3))
print("예측      :", preds)
print("정답      :", y[:10].astype(int))

In [ ]:
all_preds = (model_bce.predict(X, verbose=0) > 0.5).astype("int32").flatten()
manual_acc = (all_preds == y.astype(int)).mean()
keras_acc = model_bce.evaluate(X, y, verbose=0)[1]
print(f"manual : {manual_acc:.4f}")
print(f"keras  : {keras_acc:.4f}")

## 마무리 — 체크리스트
- [ ] 470개 데이터로 두 손실 함수 모두 학습 성공
- [ ] BCE 손실이 MSE보다 빠르게 수렴하는지 (loss curve 비교)
- [ ] 시그모이드 출력 → 0.5 임계 → 클래스 변환 직접 해보기
- [ ] 정확도가 양 클래스 분포(`np.bincount(y)`)와 어떤 관계인지 (불균형 데이터)